# FastSpeech 2 Text-to-Speech (English & Telugu) — Run 2 Recovery Pipeline (`R0`–`R5`)

### Why Run 2 Exists & How It Protects Your Work
- **Root Cause Fixed in Code (`b7718dd+`):** MFA 2.2.17 writes interior pauses as empty intervals `""` in `.TextGrid` files. In Run 1, `tgt.io.read_textgrid()` defaulted to `include_empty_intervals=False`, silently dropping interior pauses and shifting every phoneme after the first pause onto earlier audio frames.
- **Strict Run Isolation (`R0`):** Run 1 archives (`English_preprocessed.tar.gz`, `Telugu_preprocessed.tar.gz`), checkpoints, logs, and gibberish samples in `/content/drive/MyDrive/FastSpeech2/` are preserved **untouched** as diagnostic evidence for your report. All Run 2 files are saved under `/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2/`.
- **Fast Alignment Reuse (`R1`):** We reuse the verified `.TextGrid` files (`4,207` English, `2,615` Telugu) from Run 1 **without re-running MFA**, regenerate the matching raw waveforms (`seed=42`), verify manifest/hash equality, and rebuild clean features with interior pauses (`sp`), pre-slice timeline checks, and train-only quantization bounds.
- **Staged Quality Gates (`R2` -> `R3` -> `R4` -> `R5`):**
  - **R0:** Mount Drive, pull latest repo, inventory Run 1, initialize `/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2/`.
  - **R1:** Restore only `.TextGrid` + `lexicon` from Run 1, regenerate raw waveforms (`seed=42`), verify manifests match, run fixed `preprocess.py`, verify data integrity, and save `*_preprocessed_v2.tar.gz`.
  - **R2:** Vocoder & frontend gate in isolated subprocesses (listen to real-mel -> HiFi-GAN reconstructions for English and both Telugu voices).
  - **R3:** Small 64-clip overfit test & early held-out pilot (listen to free-running speech with predicted durations/pitch/energy before spending full compute).
  - **R4:** Full independent production runs from fresh weights (`restore_step=0` in `pause_fix_v2`) + checkpoint listening sweep.
  - **R5:** Synthesize all 10 required test sentences (`E1–E4`, `T1–T4`), generate 30 variance adaptor plots + 4 loss curves, review audio, and package `Results_Deliverables_v2.zip`.

## Section R0: Mount Drive, Update Repo, Inventory Run 1 & Initialize Run 2 (`pause_fix_v2`)

In [ ]:
import os, sys, json, glob, shutil, hashlib, subprocess
from datetime import datetime
os.environ["MPLBACKEND"] = "Agg"

from google.colab import drive
drive.mount("/content/drive")

RUN1_ROOT = "/content/drive/MyDrive/FastSpeech2"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
REPO_URL = "https://github.com/SandeshAwate6055/FastSpeech2-English-Telugu.git"
REPO_DIR = "/content/FastSpeech2-English-Telugu"

# Create isolated Run 2 directory hierarchy
for sub in [
    "diagnostics",
    "diagnostics/vocoder_gate",
    "diagnostics/overfit64",
    "diagnostics/ckpt_sweep",
    "English/checkpoints",
    "English/logs",
    "Telugu/checkpoints",
    "Telugu/logs",
    "Results/English",
    "Results/Telugu",
]:
    os.makedirs(os.path.join(RUN2_ROOT, sub), exist_ok=True)

# Clone or update repository to latest commit
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

git_commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print(f"Checked out repository commit: {git_commit}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-c",
    "import nltk; nltk.download('averaged_perceptron_tagger', quiet=True); "
    "nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('cmudict', quiet=True)"
], check=True)

# Ensure Universal HiFi-GAN vocoder weights exist in both language folders
for lang in ["English", "Telugu"]:
    hifi_ckpt = os.path.join(REPO_DIR, lang, "hifigan", "generator_universal.pth.tar")
    if not os.path.exists(hifi_ckpt):
        if not os.path.exists("/tmp/ming_fs2"):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ming024/FastSpeech2.git", "/tmp/ming_fs2"], check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/ming_fs2/hifigan/generator_universal.pth.tar.zip", "-d", os.path.join(REPO_DIR, lang, "hifigan")], check=True)
        print(f"Extracted generator_universal.pth.tar for {lang}")

# Inventory Run 1 artifacts without modifying them
run1_inventory = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "git_commit": git_commit,
    "run1_root": RUN1_ROOT,
    "run2_root": RUN2_ROOT,
    "archives": {},
    "languages": {},
}
for lang in ["English", "Telugu"]:
    arc = os.path.join(RUN1_ROOT, f"{lang}_preprocessed.tar.gz")
    run1_inventory["archives"][lang] = {
        "path": arc,
        "exists": os.path.exists(arc),
        "size_mb": round(os.path.getsize(arc) / (1024 * 1024), 2) if os.path.exists(arc) else 0,
    }
    ckpts = sorted(glob.glob(os.path.join(RUN1_ROOT, lang, "checkpoints", "*.pth.tar")))
    wavs = sorted(glob.glob(os.path.join(RUN1_ROOT, "Results", lang, "test_generated_audios", "*.wav")))
    run1_inventory["languages"][lang] = {
        "checkpoints": [os.path.basename(c) for c in ckpts],
        "train_log_exists": os.path.exists(os.path.join(RUN1_ROOT, lang, "logs", "train_losses.csv")),
        "val_log_exists": os.path.exists(os.path.join(RUN1_ROOT, lang, "logs", "val_losses.csv")),
        "run1_test_wavs": [os.path.basename(w) for w in wavs],
    }

inv_path = os.path.join(RUN2_ROOT, "diagnostics", "run1_inventory.json")
with open(inv_path, "w", encoding="utf-8") as f:
    json.dump(run1_inventory, f, indent=2)

print("\n=== Run 1 Preservation & Inventory Summary ===")
print(json.dumps(run1_inventory, indent=2))


## Section R1: Restore Only Verified `.TextGrid` Alignments, Regenerate Raw Audio (`seed=42`), Rebuild Corrected Features & Audit Integrity
- **Does NOT re-run MFA** (saves ~45 minutes): extracts only `preprocessed_data/TextGrid` and `lexicon` from the immutable Run 1 archives into staging.
- Regenerates `raw_data` waveforms (`seed=42`), verifies that utterance IDs, speakers, and SHA-1 hashes match the Run 1 manifests, then runs the fixed `preprocess.py` (`include_empty_intervals=True` + full timeline span checks + train-only quantization bounds).
- Saves new validated archives `English_preprocessed_v2.tar.gz` and `Telugu_preprocessed_v2.tar.gz` inside `/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2/`.

In [ ]:
import os, sys, json, glob, shutil, tarfile, subprocess
import numpy as np
from getpass import getpass

RUN1_ROOT = "/content/drive/MyDrive/FastSpeech2"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
REPO_DIR = "/content/FastSpeech2-English-Telugu"
STAGING_DIR = "/content/run1_staging"

FORCE_REBUILD_V2 = False  # Set True if you ever want to force rebuilding v2 archives from scratch

en_v2_archive = os.path.join(RUN2_ROOT, "English_preprocessed_v2.tar.gz")
te_v2_archive = os.path.join(RUN2_ROOT, "Telugu_preprocessed_v2.tar.gz")

def parse_manifest(path):
    rows = {}
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) >= 3:
                utt_id, spk_id, text = parts[0], parts[1], parts[2]
                dur = float(parts[3]) if len(parts) >= 4 else None
                sha1 = parts[4] if len(parts) >= 5 else None
                rows[utt_id] = {"spk": spk_id, "text": text, "dur": dur, "sha1": sha1}
    return rows

for lang, v2_arc in [("English", en_v2_archive), ("Telugu", te_v2_archive)]:
    lang_dir = os.path.join(REPO_DIR, lang)
    if os.path.exists(v2_arc) and not FORCE_REBUILD_V2:
        print(f"\n[{lang}] Found existing validated Run 2 archive at {v2_arc}. Extracting to {lang_dir} ...")
        for stale in ["preprocessed_data", "raw_data"]:
            p = os.path.join(lang_dir, stale)
            if os.path.exists(p):
                shutil.rmtree(p)
        subprocess.run(["tar", "-xzf", v2_arc, "-C", lang_dir], check=True)
        # Also ensure a few raw wavs exist for R2 vocoder gate if not in archive
        continue

    run1_arc = os.path.join(RUN1_ROOT, f"{lang}_preprocessed.tar.gz")
    if not os.path.exists(run1_arc):
        raise FileNotFoundError(f"Missing Run 1 archive: {run1_arc}")

    print(f"\n==================== [{lang}] Stage R1: Alignment-Only Restore & Rebuild ====================")
    lang_stage = os.path.join(STAGING_DIR, lang)
    if os.path.exists(lang_stage):
        shutil.rmtree(lang_stage)
    os.makedirs(lang_stage, exist_ok=True)

    print(f"1. Extracting Run 1 archive {run1_arc} to temporary staging {lang_stage} ...")
    subprocess.run(["tar", "-xzf", run1_arc, "-C", lang_stage], check=True)

    # Clean local preprocessed_data & raw_data in repo so no stale Run 1 features remain
    for clean_sub in ["preprocessed_data", "raw_data"]:
        target_sub = os.path.join(lang_dir, clean_sub)
        if os.path.exists(target_sub):
            shutil.rmtree(target_sub)
    os.makedirs(os.path.join(lang_dir, "preprocessed_data"), exist_ok=True)

    # Copy ONLY TextGrid and lexicon from staging
    staged_tg = os.path.join(lang_stage, "preprocessed_data", "TextGrid")
    if not os.path.exists(staged_tg):
        raise RuntimeError(f"No TextGrid directory found inside {run1_arc}")
    shutil.copytree(staged_tg, os.path.join(lang_dir, "preprocessed_data", "TextGrid"))

    staged_lex = os.path.join(lang_stage, "lexicon")
    if os.path.exists(staged_lex):
        if os.path.exists(os.path.join(lang_dir, "lexicon")):
            shutil.rmtree(os.path.join(lang_dir, "lexicon"))
        shutil.copytree(staged_lex, os.path.join(lang_dir, "lexicon"))

    tg_count = sum(len(files) for _, _, files in os.walk(os.path.join(lang_dir, "preprocessed_data", "TextGrid")) if any(f.endswith(".TextGrid") for f in files))
    print(f"   Restored {tg_count} .TextGrid files (and lexicon) into {lang_dir}. Old mel/pitch/energy/duration discarded.")

    # 2. Regenerate raw waveforms with seed=42
    if lang == "English":
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            if not os.environ.get("HF_TOKEN"):
                os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face Read Token (hf_...) for ai4bharat/Svarah: ")

    print(f"2. Running prepare_data.py --train_hours 5.0 --val_hours 1.0 --seed 42 for {lang} ...")
    subprocess.run(
        [sys.executable, "prepare_data.py", "--train_hours", "5.0", "--val_hours", "1.0", "--seed", "42"],
        cwd=lang_dir,
        check=True,
    )

    # 3. Compare regenerated manifests against staged Run 1 manifests
    for split_name in ["train_filelist.txt", "val_filelist.txt"]:
        old_m = parse_manifest(os.path.join(lang_stage, "raw_data", split_name))
        new_m = parse_manifest(os.path.join(lang_dir, "raw_data", split_name))
        if old_m:
            old_ids = set(old_m.keys())
            new_ids = set(new_m.keys())
            overlap = len(old_ids & new_ids)
            spk_mismatches = sum(1 for k in (old_ids & new_ids) if old_m[k]["spk"] != new_m[k]["spk"])
            sha_mismatches = sum(
                1 for k in (old_ids & new_ids)
                if old_m[k]["sha1"] and new_m[k]["sha1"] and old_m[k]["sha1"] != new_m[k]["sha1"]
            )
            print(f"   Manifest check [{split_name}]: old={len(old_ids)}, new={len(new_ids)}, overlap={overlap}, spk_mismatch={spk_mismatches}, sha1_mismatch={sha_mismatches}")
            if overlap < len(old_ids) or spk_mismatches > 0 or sha_mismatches > 0:
                raise RuntimeError(f"Manifest mismatch detected for {lang} {split_name}! Stopping before applying alignments.")

    # 4. Run hardened preprocess.py (with include_empty_intervals=True, pre-slice timeline checks, train-only bounds)
    print(f"3. Running hardened preprocess.py --config config.yaml for {lang} ...")
    subprocess.run(
        [sys.executable, "preprocess.py", "--config", "config.yaml"],
        cwd=lang_dir,
        check=True,
    )

    # 5. Archive validated v2 features (including reserve filelists + provenance)
    print(f"4. Archiving rebuilt v2 features to {v2_arc} ...")
    files_to_archive = [
        "preprocessed_data",
        "lexicon",
        "raw_data/data_stats.json",
        "raw_data/train_filelist.txt",
        "raw_data/val_filelist.txt",
    ]
    for opt_f in ["raw_data/reserve_train_filelist.txt", "raw_data/reserve_val_filelist.txt"]:
        if os.path.exists(os.path.join(lang_dir, opt_f)):
            files_to_archive.append(opt_f)
    subprocess.run(["tar", "-czf", v2_arc, "-C", lang_dir] + files_to_archive, check=True)
    print(f"   Saved {v2_arc} ({os.path.getsize(v2_arc)/(1024*1024):.2f} MB)")

    if os.path.exists(lang_stage):
        shutil.rmtree(lang_stage)


In [ ]:
# R1 Integrity & Pause-Recovery Audit across English & Telugu
import os, json, numpy as np

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"

audit_report = {}
for lang in ["English", "Telugu"]:
    pdir = os.path.join(REPO_DIR, lang, "preprocessed_data")
    rdir = os.path.join(REPO_DIR, lang, "raw_data")
    summary_path = os.path.join(pdir, "preprocess_summary.json")
    stats_path = os.path.join(pdir, "stats.json")
    speakers_path = os.path.join(pdir, "speakers.json")

    summary = json.load(open(summary_path, "r", encoding="utf-8")) if os.path.exists(summary_path) else {}
    stats = json.load(open(stats_path, "r", encoding="utf-8"))
    speakers = json.load(open(speakers_path, "r", encoding="utf-8"))

    split_info = {}
    train_utts, val_utts = set(), set()
    train_spks, val_spks = set(), set()

    for split in ["train", "val"]:
        lines = [l.strip() for l in open(os.path.join(pdir, f"{split}.txt"), encoding="utf-8") if l.strip()]
        len_mismatch = 0
        mel_mismatch = 0
        nonfinite = 0
        utts_with_sp = 0
        total_sp_tokens = 0
        total_tokens = 0
        sp_loudness, vowel_loudness = [], []
        phone_loud = {}
        all_loud, all_eng = [], []
        total_frames = 0

        for line in lines:
            utt_id, spk, ph_str, _ = line.split("|")[:4]
            phones = ph_str.strip("{}").split()
            if split == "train":
                train_utts.add(utt_id)
                train_spks.add(spk)
            else:
                val_utts.add(utt_id)
                val_spks.add(spk)

            dur = np.load(os.path.join(pdir, "duration", f"{spk}-duration-{utt_id}.npy"))
            pitch = np.load(os.path.join(pdir, "pitch", f"{spk}-pitch-{utt_id}.npy"))
            energy = np.load(os.path.join(pdir, "energy", f"{spk}-energy-{utt_id}.npy"))
            mel = np.load(os.path.join(pdir, "mel", f"{spk}-mel-{utt_id}.npy"))

            if not (len(phones) == len(dur) == len(pitch) == len(energy)):
                len_mismatch += 1
            if int( np.sum(dur) ) != int(mel.shape[0]):
                mel_mismatch += 1
            if not (np.isfinite(dur).all() and np.isfinite(pitch).all() and np.isfinite(energy).all() and np.isfinite(mel).all()):
                nonfinite += 1

            total_frames += int(mel.shape[0])
            total_tokens += len(phones)
            if "sp" in phones:
                utts_with_sp += 1
                total_sp_tokens += sum(1 for p in phones if p == "sp")

            idx = 0
            for p, d, e in zip(phones, dur, energy):
                d = int(d)
                if d > 0:
                    loud = float(mel[idx:idx+d].mean())
                    all_loud.append(loud)
                    all_eng.append(float(e))
                    phone_loud.setdefault(p, []).append(loud)
                    if p in ("sp", "sil", "spn"):
                        sp_loudness.append(loud)
                    else:
                        vowel_loudness.append(loud)
                idx += d

        grand_mean = np.mean(all_loud)
        ss_tot = sum((x - grand_mean) ** 2 for x in all_loud)
        ss_bet = sum(len(v) * (np.mean(v) - grand_mean) ** 2 for v in phone_loud.values())
        r2_mel_loud = float(ss_bet / ss_tot) if ss_tot > 0 else 0.0
        corr_eng = float(np.corrcoef(all_eng, all_loud)[0, 1]) if len(all_eng) > 1 else 0.0
        hours = (total_frames * 256.0 / 22050.0) / 3600.0

        split_info[split] = {
            "utts": len(lines),
            "hours": round(hours, 3),
            "len_mismatch": len_mismatch,
            "mel_mismatch": mel_mismatch,
            "nonfinite": nonfinite,
            "utts_with_sp": utts_with_sp,
            "utts_with_sp_pct": round(100.0 * utts_with_sp / max(1, len(lines)), 1),
            "total_sp_tokens": total_sp_tokens,
            "mean_sp_loudness": round(float(np.mean(sp_loudness)), 3) if sp_loudness else None,
            "mean_non_sp_loudness": round(float(np.mean(vowel_loudness)), 3) if vowel_loudness else None,
            "corr_saved_energy_mel_loudness": round(corr_eng, 3),
            "r2_mel_loudness_by_phone": round(r2_mel_loud, 3),
        }

    utt_overlap = len(train_utts & val_utts)
    unseen_val_spks = len(val_spks - train_spks)
    audit_report[lang] = {
        "num_speakers": len(speakers),
        "utt_overlap": utt_overlap,
        "unseen_val_speakers": unseen_val_spks,
        "stats_json": stats,
        "preprocess_summary": summary,
        "splits": split_info,
    }

    print(f"\n==================== {lang} R1 Integrity Audit ====================")
    print(f"Speakers: {len(speakers)} | Train-Val Utterance Overlap: {utt_overlap} | Unseen Val Speakers: {unseen_val_spks}")
    for s in ["train", "val"]:
        si = split_info[s]
        print(
            f"  [{s.upper()}] {si['utts']} utts ({si['hours']}h) | "
            f"len_err={si['len_mismatch']} | mel_err={si['mel_mismatch']} | nonfinite={si['nonfinite']} | "
            f"utts_with_sp={si['utts_with_sp']} ({si['utts_with_sp_pct']}%, {si['total_sp_tokens']} tokens) | "
            f"sp_loud={si['mean_sp_loudness']} vs speech_loud={si['mean_non_sp_loudness']} | "
            f"corr={si['corr_saved_energy_mel_loudness']} | R2={si['r2_mel_loudness_by_phone']}"
        )
    assert utt_overlap == 0, f"{lang}: Train/Val overlap detected!"
    assert unseen_val_spks == 0, f"{lang}: Validation contains speakers not in training!"
    assert split_info["train"]["len_mismatch"] == 0 and split_info["val"]["len_mismatch"] == 0
    assert split_info["train"]["mel_mismatch"] == 0 and split_info["val"]["mel_mismatch"] == 0
    assert split_info["train"]["nonfinite"] == 0 and split_info["val"]["nonfinite"] == 0
    assert split_info["train"]["total_sp_tokens"] > 0, f"{lang}: Zero 'sp' pause tokens found!"

with open(os.path.join(RUN2_ROOT, "diagnostics", "r1_integrity_audit.json"), "w", encoding="utf-8") as f:
    json.dump(audit_report, f, indent=2)
print("\nSUCCESS: R1 Data Integrity & Timeline Audit passed for both languages!")


## Section R2: Isolated Vocoder & Frontend Quality Gate
- Runs in **separate per-language Python subprocesses** so `sys.modules` never mixes `English` and `Telugu` packages.
- Reconstructs multiple clips for English and both Telugu voices (`te_female`, `te_male`) from ground-truth mels via Universal HiFi-GAN (`generator_universal.pth.tar`), saves paired `orig.wav` vs `reconstructed.wav` in `runs/pause_fix_v2/diagnostics/vocoder_gate/`, and displays interactive audio players.
- Verifies frontend normalization and phoneme conversion on all 10 required test sentences (`E1–E4`, `T1–T4`, preserving exact Telugu diacritics and `E3`/`T3` double spaces).

In [ ]:
import os, sys, json, subprocess
from IPython.display import Audio, display, HTML

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
VOC_OUT_DIR = os.path.join(RUN2_ROOT, "diagnostics", "vocoder_gate")
os.makedirs(VOC_OUT_DIR, exist_ok=True)

probe_script = r'''
import os, sys, json, yaml, torch, shutil
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
out_dir = sys.argv[3]
sys.path.insert(0, lang_dir)

from utils.model import get_vocoder, vocoder_infer
from inference import TEST_SENTENCES, preprocess_english if lang == "English" else preprocess_telugu
from text.symbols import symbols

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
vocoder = get_vocoder(cfg, device)

pdir = os.path.join(lang_dir, "preprocessed_data")
rdir = os.path.join(lang_dir, "raw_data")
lines = [l.strip().split("|") for l in open(os.path.join(pdir, "val.txt"), encoding="utf-8") if l.strip()]

# Pick up to 3 clips covering distinct speakers (especially te_female and te_male for Telugu)
selected = []
seen_spk = set()
for parts in lines:
    utt_id, spk = parts[0], parts[1]
    if spk not in seen_spk or len(selected) < 3:
        selected.append((utt_id, spk, parts[3]))
        seen_spk.add(spk)
    if len(selected) >= 3 and (lang == "English" or len(seen_spk) >= 2):
        break

records = []
for utt_id, spk, raw_txt in selected[:3]:
    mel_path = os.path.join(pdir, "mel", f"{spk}-mel-{utt_id}.npy")
    mel = np.load(mel_path)
    mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).transpose(1, 2).to(device)
    wav_rec = vocoder_infer(mel_tensor, vocoder, cfg, cfg)[0]
    rec_path = os.path.join(out_dir, f"{lang}_{spk}_{utt_id}_reconstructed.wav")
    wavfile.write(rec_path, 22050, wav_rec)

    orig_src = os.path.join(rdir, spk, f"{utt_id}.wav")
    orig_dst = os.path.join(out_dir, f"{lang}_{spk}_{utt_id}_orig.wav")
    if os.path.exists(orig_src):
        shutil.copy2(orig_src, orig_dst)
    records.append({
        "utt_id": utt_id,
        "speaker": spk,
        "text": raw_txt,
        "mel_frames": int(mel.shape[0]),
        "rec_duration_s": round(len(wav_rec) / 22050.0, 2),
        "peak_abs": round(float(np.max(np.abs(wav_rec))) / 32768.0, 3),
        "orig_wav": orig_dst if os.path.exists(orig_dst) else None,
        "rec_wav": rec_path,
    })

# Verify all 5 test sentences for this language through the exact frontend
preprocess_fn = preprocess_english if lang == "English" else preprocess_telugu
frontend_checks = []
for sid, raw_text in TEST_SENTENCES:
    seq, phones, norm_text = preprocess_fn(raw_text, cfg)
    assert len(seq) == len(phones) and len(seq) > 0, f"Empty or mismatched sequence for {sid}"
    frontend_checks.append({
        "id": sid,
        "raw_text": raw_text,
        "normalized_text": norm_text,
        "num_phones": len(phones),
        "phones_preview": " ".join(phones[:25]) + (" ..." if len(phones) > 25 else ""),
    })

out_json = os.path.join(out_dir, f"{lang}_r2_gate.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump({"vocoder_samples": records, "frontend_checks": frontend_checks}, f, indent=2, ensure_ascii=False)
'''

with open("/tmp/run_r2_gate.py", "w", encoding="utf-8") as f:
    f.write(probe_script)

for lang in ["English", "Telugu"]:
    print(f"\n==================== Running Isolated R2 Gate Subprocess for {lang} ====================")
    subprocess.run([sys.executable, "/tmp/run_r2_gate.py", lang, os.path.join(REPO_DIR, lang), VOC_OUT_DIR], check=True)
    res = json.load(open(os.path.join(VOC_OUT_DIR, f"{lang}_r2_gate.json"), encoding="utf-8"))

    print(f"\n--- {lang} Frontend Check on Mandatory Test Sentences ---")
    for fc in res["frontend_checks"]:
        print(f"  [{fc['id']}] ({fc['num_phones']} phones) raw={fc['raw_text']!r}")
        print(f"       norm={fc['normalized_text']!r}")
        print(f"       phones={fc['phones_preview']}")

    print(f"\n--- {lang} Ground-Truth Mel -> Universal HiFi-GAN Reconstruction Samples ---")
    for rec in res["vocoder_samples"]:
        print(f"\nClip: {rec['utt_id']} | Speaker: {rec['speaker']} | {rec['rec_duration_s']}s | Text: {rec['text'][:90]}")
        if rec["orig_wav"] and os.path.exists(rec["orig_wav"]):
            print("  Original 22.05kHz WAV:")
            display(Audio(rec["orig_wav"]))
        print("  HiFi-GAN Reconstructed WAV from Rebuilt Mel:")
        display(Audio(rec["rec_wav"]))


## Section R3: Small Subset Overfit (64 Clips, 2,000 Steps) & Free-Running Speech Gate
- Before spending ~2.5 hours on full 60,000-step runs, we train a fast **2,000-step diagnostic overfit** on a 64-clip subset per language (~2–3 minutes on L4 GPU) in a temporary directory.
- Synthesizes **free-running speech** (using predicted duration, predicted pitch, and predicted energy—not ground-truth duration) on familiar clips and held-out clips so you can hear and verify that the model + pause-fixed features produce intelligible words.

In [ ]:
import os, sys, json, glob, shutil, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
OVERFIT_ROOT = "/content/overfit64_work"
OVERFIT_DIAG_DIR = os.path.join(RUN2_ROOT, "diagnostics", "overfit64")
os.makedirs(OVERFIT_DIAG_DIR, exist_ok=True)

OVERFIT_STEPS = 2000

overfit_eval_script = r'''
import os, sys, yaml, torch
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
prep_dir = sys.argv[3]
ckpt_dir = sys.argv[4]
out_dir = sys.argv[5]
step = int(sys.argv[6])

sys.path.insert(0, lang_dir)
from utils.model import get_model, get_vocoder, vocoder_infer
from text import text_to_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
cfg["path"]["preprocessed_path"] = prep_dir
cfg["path"]["ckpt_path"] = ckpt_dir

class DummyArgs:
    restore_step = step
    ckpt_file = os.path.join(ckpt_dir, f"{step}.pth.tar")

model = get_model(DummyArgs(), cfg, device, train=False)
vocoder = get_vocoder(cfg, device)
speakers = json.load(open(os.path.join(prep_dir, "speakers.json"), encoding="utf-8"))
cleaners = cfg["preprocessing"]["text"]["text_cleaners"]

train_lines = [l.strip().split("|") for l in open(os.path.join(prep_dir, "train.txt"), encoding="utf-8") if l.strip()]
val_lines = [l.strip().split("|") for l in open(os.path.join(lang_dir, "preprocessed_data", "val.txt"), encoding="utf-8") if l.strip()]

samples_to_synth = [
    ("seen_train_1", train_lines[0]),
    ("seen_train_2", train_lines[min(1, len(train_lines)-1)]),
    ("heldout_val_1", val_lines[0]),
]

results = []
for tag, parts in samples_to_synth:
    utt_id, spk_name, ph_str, raw_txt = parts[0], parts[1], parts[2], parts[3]
    spk_id = speakers.get(spk_name, 0)
    seq = np.array(text_to_sequence(ph_str, cleaners), dtype=np.int64)
    src = torch.from_numpy(seq).long().unsqueeze(0).to(device)
    src_len = torch.tensor([len(seq)], dtype=torch.long, device=device)
    spk_tensor = torch.tensor([spk_id], dtype=torch.long, device=device)

    with torch.no_grad():
        out = model(spk_tensor, src, src_len, max_src_len=int(len(seq)))
        mel_post = out[1].transpose(1, 2)
        dur_pred = out[4][0].detach().cpu().numpy()
        wav = vocoder_infer(mel_post, vocoder, cfg, cfg)[0]

    wav_path = os.path.join(out_dir, f"{lang}_overfit_{tag}_{utt_id}_free_running.wav")
    wavfile.write(wav_path, 22050, wav)
    results.append({
        "tag": tag,
        "utt_id": utt_id,
        "speaker": spk_name,
        "text": raw_txt,
        "pred_frames": int(np.sum(dur_pred)),
        "duration_s": round(len(wav) / 22050.0, 2),
        "wav_path": wav_path,
    })

with open(os.path.join(out_dir, f"{lang}_overfit_summary.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
'''
with open("/tmp/eval_overfit.py", "w", encoding="utf-8") as f:
    f.write("import json\n" + overfit_eval_script)

for lang in ["English", "Telugu"]:
    print(f"\n==================== [{lang}] Stage R3: 64-Clip Overfit Diagnostic ({OVERFIT_STEPS} steps) ====================")
    lang_dir = os.path.join(REPO_DIR, lang)
    orig_prep = os.path.join(lang_dir, "preprocessed_data")
    work_dir = os.path.join(OVERFIT_ROOT, lang)
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    prep_sub = os.path.join(work_dir, "preprocessed_data")
    ckpt_sub = os.path.join(work_dir, "checkpoints")
    log_sub = os.path.join(work_dir, "logs")
    res_sub = os.path.join(work_dir, "results")
    os.makedirs(prep_sub, exist_ok=True)

    # Symlink feature directories and metadata
    for item in ["mel", "pitch", "energy", "duration", "stats.json", "speakers.json"]:
        os.symlink(os.path.join(orig_prep, item), os.path.join(prep_sub, item))

    # Pick 64 single-speaker (or top-speaker) clips so the tiny diagnostic converges rapidly
    all_train = [l for l in open(os.path.join(orig_prep, "train.txt"), encoding="utf-8") if l.strip()]
    spk_counts = {}
    for l in all_train:
        spk = l.split("|")[1]
        spk_counts[spk] = spk_counts.get(spk, 0) + 1
    top_spk = max(spk_counts, key=spk_counts.get)
    top_spk_lines = [l for l in all_train if l.split("|")[1] == top_spk]
    subset64 = (top_spk_lines + [l for l in all_train if l.split("|")[1] != top_spk])[:64]

    with open(os.path.join(prep_sub, "train.txt"), "w", encoding="utf-8") as f:
        f.writelines(subset64)
    with open(os.path.join(prep_sub, "val.txt"), "w", encoding="utf-8") as f:
        f.writelines(subset64[:16])

    subprocess.run([
        sys.executable, "train.py",
        "--config", "config.yaml",
        "--restore_step", "0",
        "--total_step", str(OVERFIT_STEPS),
        "--val_step", str(OVERFIT_STEPS),
        "--save_step", str(OVERFIT_STEPS),
        "--synth_step", str(OVERFIT_STEPS),
        "--preprocessed_path", prep_sub,
        "--ckpt_path", ckpt_sub,
        "--log_path", log_sub,
        "--result_path", res_sub,
    ], cwd=lang_dir, check=True)

    subprocess.run([
        sys.executable, "/tmp/eval_overfit.py",
        lang, lang_dir, prep_sub, ckpt_sub, OVERFIT_DIAG_DIR, str(OVERFIT_STEPS)
    ], check=True)

    summary = json.load(open(os.path.join(OVERFIT_DIAG_DIR, f"{lang}_overfit_summary.json"), encoding="utf-8"))
    for item in summary:
        print(f"\n[{lang} Overfit {item['tag']}] {item['utt_id']} ({item['speaker']}) | {item['duration_s']}s | Text: {item['text'][:90]}")
        display(Audio(item["wav_path"]))


## Section R4: Full Independent Production Training (`English` & `Telugu`) in `runs/pause_fix_v2`
- **Strict Run 2 Isolation:** Checkpoints, logs, and validation plots are written exclusively to `/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2/{English,Telugu}/`.
- **Never loads Run 1 checkpoints:** Starts from fresh weights (`restore_step = 0`) unless a Run 2 checkpoint already exists inside `runs/pause_fix_v2/{Language}/checkpoints`.
- **Recommended Pilot + Full Strategy:**
  - You can first set `PILOT_OR_TOTAL_STEP = 5000` to inspect the 5,000-step pilot checkpoint with **Cell R4-C**, and then set `PILOT_OR_TOTAL_STEP = 60000` to seamlessly resume from step `5000` to `60000`!

In [ ]:
# R4-A: Train English FastSpeech 2 Model (Isolated in runs/pause_fix_v2/English)
import os, sys, glob, subprocess

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
EN_DIR = os.path.join(REPO_DIR, "English")
EN_CKPT_DIR = os.path.join(RUN2_ROOT, "English", "checkpoints")
EN_LOG_DIR = os.path.join(RUN2_ROOT, "English", "logs")
EN_RES_DIR = os.path.join(RUN2_ROOT, "Results", "English")

os.makedirs(EN_CKPT_DIR, exist_ok=True)
os.makedirs(EN_LOG_DIR, exist_ok=True)
os.makedirs(EN_RES_DIR, exist_ok=True)

# Ensure v2 preprocessed data is present locally
if not os.path.exists(os.path.join(EN_DIR, "preprocessed_data", "train.txt")):
    en_v2_arc = os.path.join(RUN2_ROOT, "English_preprocessed_v2.tar.gz")
    assert os.path.exists(en_v2_arc), f"Run Section R1 first! Missing {en_v2_arc}"
    subprocess.run(["tar", "-xzf", en_v2_arc, "-C", EN_DIR], check=True)

PILOT_OR_TOTAL_STEP = 60000  # Set to 5000 for an early pilot stop, or 60000 for full run

ckpts = [
    int(os.path.basename(f).replace(".pth.tar", ""))
    for f in glob.glob(os.path.join(EN_CKPT_DIR, "*.pth.tar"))
    if os.path.basename(f).replace(".pth.tar", "").isdigit()
]
restore_step = max(ckpts) if ckpts else 0
print(f"[English Run 2] Starting/Resuming from step {restore_step} -> {PILOT_OR_TOTAL_STEP} in {EN_CKPT_DIR} ...")

subprocess.run([
    sys.executable, "train.py",
    "--config", "config.yaml",
    "--restore_step", str(restore_step),
    "--total_step", str(PILOT_OR_TOTAL_STEP),
    "--ckpt_path", EN_CKPT_DIR,
    "--log_path", EN_LOG_DIR,
    "--result_path", EN_RES_DIR,
], cwd=EN_DIR, check=True)


In [ ]:
# R4-B: Train Telugu FastSpeech 2 Model (Isolated in runs/pause_fix_v2/Telugu)
import os, sys, glob, subprocess

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
TE_DIR = os.path.join(REPO_DIR, "Telugu")
TE_CKPT_DIR = os.path.join(RUN2_ROOT, "Telugu", "checkpoints")
TE_LOG_DIR = os.path.join(RUN2_ROOT, "Telugu", "logs")
TE_RES_DIR = os.path.join(RUN2_ROOT, "Results", "Telugu")

os.makedirs(TE_CKPT_DIR, exist_ok=True)
os.makedirs(TE_LOG_DIR, exist_ok=True)
os.makedirs(TE_RES_DIR, exist_ok=True)

# Ensure v2 preprocessed data is present locally
if not os.path.exists(os.path.join(TE_DIR, "preprocessed_data", "train.txt")):
    te_v2_arc = os.path.join(RUN2_ROOT, "Telugu_preprocessed_v2.tar.gz")
    assert os.path.exists(te_v2_arc), f"Run Section R1 first! Missing {te_v2_arc}"
    subprocess.run(["tar", "-xzf", te_v2_arc, "-C", TE_DIR], check=True)

PILOT_OR_TOTAL_STEP = 60000  # Set to 5000 for an early pilot stop, or 60000 for full run

ckpts = [
    int(os.path.basename(f).replace(".pth.tar", ""))
    for f in glob.glob(os.path.join(TE_CKPT_DIR, "*.pth.tar"))
    if os.path.basename(f).replace(".pth.tar", "").isdigit()
]
restore_step = max(ckpts) if ckpts else 0
print(f"[Telugu Run 2] Starting/Resuming from step {restore_step} -> {PILOT_OR_TOTAL_STEP} in {TE_CKPT_DIR} ...")

subprocess.run([
    sys.executable, "train.py",
    "--config", "config.yaml",
    "--restore_step", str(restore_step),
    "--total_step", str(PILOT_OR_TOTAL_STEP),
    "--ckpt_path", TE_CKPT_DIR,
    "--log_path", TE_LOG_DIR,
    "--result_path", TE_RES_DIR,
], cwd=TE_DIR, check=True)


In [ ]:
# R4-C: Checkpoint Validation Loss Table & Free-Running Audio Sweep (Select Best Checkpoint by Ear + Val Loss)
import os, sys, csv, json, glob, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"
SWEEP_DIR = os.path.join(RUN2_ROOT, "diagnostics", "ckpt_sweep")
os.makedirs(SWEEP_DIR, exist_ok=True)

sweep_script = r'''
import os, sys, json, glob, yaml, torch
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
ckpt_dir = sys.argv[3]
out_dir = sys.argv[4]

sys.path.insert(0, lang_dir)
from utils.model import get_model, get_vocoder, vocoder_infer
from text import text_to_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
cfg["path"]["ckpt_path"] = ckpt_dir
vocoder = get_vocoder(cfg, device)

pdir = os.path.join(lang_dir, "preprocessed_data")
speakers = json.load(open(os.path.join(pdir, "speakers.json"), encoding="utf-8"))
cleaners = cfg["preprocessing"]["text"]["text_cleaners"]
val_lines = [l.strip().split("|") for l in open(os.path.join(pdir, "val.txt"), encoding="utf-8") if l.strip()]

# Pick 2 representative held-out validation sentences
heldout = val_lines[:2]

# Candidate checkpoints to compare
all_ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.pth.tar")))
candidates = []
for name in ["best_val.pth.tar", "10000.pth.tar", "20000.pth.tar", "30000.pth.tar", "40000.pth.tar", "50000.pth.tar", "60000.pth.tar"]:
    p = os.path.join(ckpt_dir, name)
    if os.path.exists(p):
        candidates.append(p)
if not candidates and all_ckpts:
    candidates = all_ckpts[-3:]

records = []
for ckpt_path in candidates:
    cname = os.path.basename(ckpt_path)
    raw_ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    step_num = raw_ckpt.get("step", cname)
    val_loss = raw_ckpt.get("val_loss", None)

    class DummyArgs:
        restore_step = 0
        ckpt_file = ckpt_path

    model = get_model(DummyArgs(), cfg, device, train=False)
    sample_wavs = []
    for idx, parts in enumerate(heldout):
        utt_id, spk_name, ph_str, raw_txt = parts[0], parts[1], parts[2], parts[3]
        spk_id = speakers.get(spk_name, 0)
        seq = np.array(text_to_sequence(ph_str, cleaners), dtype=np.int64)
        src = torch.from_numpy(seq).long().unsqueeze(0).to(device)
        src_len = torch.tensor([len(seq)], dtype=torch.long, device=device)
        spk_tensor = torch.tensor([spk_id], dtype=torch.long, device=device)
        with torch.no_grad():
            out = model(spk_tensor, src, src_len, max_src_len=int(len(seq)))
            mel_post = out[1].transpose(1, 2)
            wav = vocoder_infer(mel_post, vocoder, cfg, cfg)[0]
        wpath = os.path.join(out_dir, f"{lang}_{cname.replace('.pth.tar','')}_{utt_id}.wav")
        wavfile.write(wpath, 22050, wav)
        sample_wavs.append({"utt_id": utt_id, "speaker": spk_name, "text": raw_txt, "wav": wpath, "dur_s": round(len(wav)/22050.0, 2)})

    records.append({"checkpoint": cname, "step": step_num, "val_loss": val_loss, "samples": sample_wavs})

with open(os.path.join(out_dir, f"{lang}_sweep.json"), "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)
'''
with open("/tmp/run_ckpt_sweep.py", "w", encoding="utf-8") as f:
    f.write(sweep_script)

for lang in ["English", "Telugu"]:
    print(f"\n==================== {lang} Run 2 Validation History & Checkpoint Sweep ====================")
    val_csv = os.path.join(RUN2_ROOT, lang, "logs", "val_losses.csv")
    if os.path.exists(val_csv):
        with open(val_csv, "r", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        print(f"Total validation evaluations logged: {len(rows)}")
        for r in rows[::max(1, len(rows)//10)] + (rows[-1:] if rows else []):
            print(f"  step={r['step']:>6} | total={float(r['total_loss']):.4f} | mel={float(r['mel_loss']):.4f} | post={float(r['mel_postnet_loss']):.4f} | pitch={float(r['pitch_loss']):.4f} | energy={float(r['energy_loss']):.4f} | dur={float(r['duration_loss']):.4f}")

    ckpt_dir = os.path.join(RUN2_ROOT, lang, "checkpoints")
    if glob.glob(os.path.join(ckpt_dir, "*.pth.tar")):
        subprocess.run([sys.executable, "/tmp/run_ckpt_sweep.py", lang, os.path.join(REPO_DIR, lang), ckpt_dir, SWEEP_DIR], check=True)
        sweep = json.load(open(os.path.join(SWEEP_DIR, f"{lang}_sweep.json"), encoding="utf-8"))
        for rec in sweep:
            print(f"\n--- {lang} Checkpoint: {rec['checkpoint']} (step={rec['step']}, val_loss={rec['val_loss']}) ---")
            for s in rec["samples"]:
                print(f"  Held-out {s['utt_id']} ({s['speaker']}, {s['dur_s']}s): {s['text'][:90]}")
                display(Audio(s["wav"]))


## Section R5: Synthesize All 10 Mandatory Test Sentences (`E1–E4`, `T1–T4`), Generate 30 Variance Plots, Listen & Package Deliverables
- Synthesizes the exact 10 test sentences using **predicted duration, predicted pitch, and predicted energy only**.
- Preserves intact Telugu vowel signs and intentional broken-word double spaces (`E3`: `"rhyth  m"`, `T3`: `"అడ  విలో"`).
- Saves all 10 WAVs, 30 variance adaptor plots (`duration`, `pitch`, `energy`), 4 loss plots, and `synthesis_summary.json` into `/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2/Results/` and packages `Results_Deliverables_v2.zip`.

In [ ]:
import os, sys, json, glob, subprocess
from IPython.display import Audio, display

REPO_DIR = "/content/FastSpeech2-English-Telugu"
RUN2_ROOT = "/content/drive/MyDrive/FastSpeech2/runs/pause_fix_v2"

# By default, inference.py prefers best_val.pth.tar if SELECTED_*_CKPT is None, or specify e.g. "60000.pth.tar" / "best_val.pth.tar"
SELECTED_EN_CKPT = None  # e.g. os.path.join(RUN2_ROOT, "English/checkpoints/best_val.pth.tar")
SELECTED_TE_CKPT = None  # e.g. os.path.join(RUN2_ROOT, "Telugu/checkpoints/best_val.pth.tar")

# Speaker selection: -1 selects top training speaker for English; 0 selects te_female (or 1 for te_male) for Telugu
EN_SPEAKER_ID = -1
TE_SPEAKER_ID = 0

for lang, sel_ckpt, spk_id in [
    ("English", SELECTED_EN_CKPT, EN_SPEAKER_ID),
    ("Telugu", SELECTED_TE_CKPT, TE_SPEAKER_ID),
]:
    lang_dir = os.path.join(REPO_DIR, lang)
    ckpt_dir = os.path.join(RUN2_ROOT, lang, "checkpoints")
    res_dir = os.path.join(RUN2_ROOT, "Results", lang)
    os.makedirs(res_dir, exist_ok=True)

    cmd = [
        sys.executable, "inference.py",
        "--config", "config.yaml",
        "--speaker_id", str(spk_id),
        "--ckpt_path", ckpt_dir,
        "--result_path", res_dir,
    ]
    if sel_ckpt:
        cmd.extend(["--ckpt_file", sel_ckpt])

    print(f"\n==================== Running Final R5 Inference for {lang} ====================")
    subprocess.run(cmd, cwd=lang_dir, check=True)

    summary_path = os.path.join(res_dir, "synthesis_summary.json")
    summary = json.load(open(summary_path, encoding="utf-8"))
    for item in summary:
        print(f"\n[{lang} - {item['id']}] Speaker: {item['speaker_name']} (ID {item['speaker_id']}) | Duration: {item['duration_seconds']:.2f}s ({item['total_mel_frames']} frames)")
        print(f"  Raw Input : {item['raw_text']}")
        print(f"  Normalized: {item['normalized_text']}")
        print(f"  Phonemes ({len(item['phonemes'])}): {' '.join(item['phonemes'][:30])}{' ...' if len(item['phonemes'])>30 else ''}")
        wav_file = os.path.join(res_dir, "test_generated_audios", f"{item['id']}.wav")
        display(Audio(wav_file))

# Verify completeness of all deliverables in RUN2_ROOT/Results
for lang in ["English", "Telugu"]:
    rdir = os.path.join(RUN2_ROOT, "Results", lang)
    wavs = glob.glob(os.path.join(rdir, "test_generated_audios", "*.wav"))
    durs = glob.glob(os.path.join(rdir, "variance_adaptor_plots", "duration", "*.png"))
    pitches = glob.glob(os.path.join(rdir, "variance_adaptor_plots", "pitch", "*.png"))
    energies = glob.glob(os.path.join(rdir, "variance_adaptor_plots", "energy", "*.png"))
    t_loss = os.path.exists(os.path.join(rdir, "training_loss.png"))
    v_loss = os.path.exists(os.path.join(rdir, "validation_loss.png"))
    print(f"\n[{lang} Deliverable Check] WAVs={len(wavs)}/5 | Duration Plots={len(durs)}/5 | Pitch Plots={len(pitches)}/5 | Energy Plots={len(energies)}/5 | Loss Curves={t_loss and v_loss}")
    assert len(wavs) == 5 and len(durs) == 5 and len(pitches) == 5 and len(energies) == 5, f"Incomplete deliverables for {lang}!"

# Package Results_Deliverables_v2.zip (including Results, logs, stats, and summaries)
zip_path = os.path.join(RUN2_ROOT, "Results_Deliverables_v2.zip")
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(
    ["zip", "-r", zip_path, "Results", "diagnostics", "English/logs", "Telugu/logs"],
    cwd=RUN2_ROOT,
    check=True,
)
print(f"\nSUCCESS: Packaged all Run 2 deliverables to {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)!")
